In [58]:
import numpy as np
import pandas as pd
import re
import string

In [59]:
def remove_punctuations(text):
    for punctuation in string.punctuation:
        text = text.replace(punctuation, '')
    return text

In [60]:
#load the model into pipiline
import pickle
with open('../static/model/model.pickle', 'rb') as f:
    model = pickle.load(f)

In [61]:
with open('../static/model/corpora/stopwords/english', 'r') as file:
    stop_words = file.read().splitlines()

In [62]:
vocab = pd.read_csv('../static/model/vocabulary.text', header=None)
tokens = vocab[0].tolist()

In [63]:
from nltk.stem import PorterStemmer
ps = PorterStemmer()

In [64]:
def preprocessing(text):
    data = pd.DataFrame([text],columns=['tweet'])
    data['tweet'] = data['tweet'].apply(lambda x: " ".join(x.lower () for x in x.split()))
    data['tweet'] = data['tweet'].apply(lambda x: " ".join(re.sub(r'^https?:\/\/.*[\r\n]*', '',x,flags=re.MULTILINE) for x in x.split()))
    data['tweet'] = data['tweet'].str.replace('\d+', '', regex = True)
    data['tweet'] = data['tweet'].apply(remove_punctuations)
    data['tweet'] = data['tweet'].apply(lambda x: " ".join (x for x in x.split() if x not in stop_words))
    data['tweet'] = data['tweet'].apply(lambda x: " ".join(ps.stem(x) for x in x.split()))
    return data['tweet']
preprocessed_txt = preprocessing(text)

<>:5: SyntaxWarning: invalid escape sequence '\d'
<>:5: SyntaxWarning: invalid escape sequence '\d'
C:\Users\User\AppData\Local\Temp\ipykernel_22432\377544002.py:5: SyntaxWarning: invalid escape sequence '\d'
  data['tweet'] = data['tweet'].str.replace('\d+', '', regex = True)


In [65]:
def vectorizer(data_set, vocabulary):
    vectorized_list = []

    for sentence in data_set:
        sentence_list = np.zeros(len(vocabulary))

        for i in range (len(vocabulary)):
            if vocabulary[i] in sentence.split():
                sentence_list[i] = 1
        vectorized_list.append(sentence_list)
    vectorized_list_new = np.asarray(vectorized_list, dtype=np.float32)

    return vectorized_list_new
vectorized_txt = vectorizer(preprocessed_txt,tokens)

# Prediction

In [116]:
def get_prediction (vectorized_txt):
    prediction = model.predict(vectorized_txt)
    if prediction == 0:
        return 'positive'
    else:
        return 'negative'

In [118]:
txt = "awesome product. i love it"
prprocessed_txt = preprocessing(txt)
vectorized_txt = vectorizer(preprocessed_txt,tokens)
prediction = get_prediction (vectorized_txt)
prediction

'positive'